# Dataset parser
The aim to this jupyter notebook is to guide thorught the process taken to adopt the raw scraped documents to the model that will be fed to BERT

In [ ]:
#Define the imports
import json, re, validators, sys
from bs4 import BeautifulSoup 
import os, chardet, magic

<b>Define the file explorers and handlers </b>

In [ ]:


#Identify the files by checking file header with magic
def guess_file_type(file_path):
    try:
        mime = magic.Magic()
        file_type = mime.from_file(file_path)
        return file_type
    except Exception as e:
        print(f"Error guessing file type for {file_path}: {str(e)}")
        return None

def detect_encoding(file):
    """
    Detects the encoding of a file using the chardet library.

    Parameters:
    file (str): The path to the file whose encoding needs to be detected.

    Returns:
    dict: A dictionary containing encoding information, including 'encoding' (the detected encoding),
          'confidence' (confidence score of the detected encoding), and 'language' (detected language).
    """
    detector = chardet.universaldetector.UniversalDetector()
    with open(file, "rb") as f:
        for line in f:
            detector.feed(line)
            if detector.done:
                break
    detector.close()
    return detector.result

def read_file(file_name):
    """
    Reads the content of a file with auto-detected encoding.

    :param file_name: The path to the file to be read.
    :type file_name: str

    :return: The contents of the file as a string.
    :rtype: str

    :raises FileNotFoundError: If the specified file does not exist.
    """
    encoding = detect_encoding(file_name)['encoding']
    try:
        with open(file_name, 'r', encoding=encoding) as file:
            return file.read()
    except FileNotFoundError:
        print(f"File not found: {file_name}")
        sys.exit(1)

def is_html(content):
    # Check if the content contains the <html> tag (case-insensitive)
    return "<html" in content.lower()

def delete_non_html_files(directory):
    for root, dirs, files in os.walk(directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            if os.path.isfile(file_path):
                try:
                    if not 'HTML' in guess_file_type(file_path) and not file_path.endswith('.csv'):
                        os.remove(file_path)
                        print(f"Deleted non-HTML file: {file_path}")
                    else:
                        if not file_path.endswith('.html'):
                            os.rename(file_path, file_path + ".html")
                except Exception as e:
                    print(f"Error processing {file_path}: {str(e)}")

#Try guess file type
#print(guess_file_type('./examples/1776/2014-05-08/itemhandler?iditems=2751'))
#print(guess_file_type('./examples/1776/2014-05-08/Main'))
#print(guess_file_type('./examples/1776/2014-05-12/image?size=400&item=2763'))

start_directory = './examples/'

if os.path.exists(start_directory):
    delete_non_html_files(start_directory)
    print("Non-HTML files deleted.")
else:
    print("The specified directory does not exist.")


### Functions to filer

In [ ]:
##Extractor class
class Extractor:
    content_raw = None
    html = None # Variable to store the HTML content,

    def __init__(self,html,encoding='utf-8') -> None:
        self.html = BeautifulSoup(html, 'html.parser')

    
    def extract_properties(self):
        """
        Extracts and assigns the title and description properties from the HTML content.

        This method finds and assigns the title and description properties from the HTML content to
        instance variables 'title' and 'description'. If the 'meta' tag with 'name' attribute equal
        to 'description' contains a 'content' attribute, it will be assigned as the description;
        otherwise, an empty string will be assigned.

        :return: None
        """
        
        self.title = self.html.find('title').text if self.html.find('title') else "No title"
        self.description = self.html.find('meta', attrs={'name': 'description'})
        if "content" in str(self.description):
            self.description = self.description.get("content")
        else:
            self.description = ""

    
    def extract_content_raw(self):
        if self.content_raw is None:
            self.content_raw = self.html.get_text(separator=" ")
            self.extract_properties()
    
            
    def extract_meta_tags(self):
        """Retrieve all meta elements from HTML object.

        Returns:
            list: List containing content from meta tags
        """
        meta_tags = self.html.find_all('meta')
        self.meta = []
        for tag in meta_tags:
            self.meta.append(tag.attrs)
        

###String parsers
def validate_link(link):
    if not isinstance(link, str):
        return False
    return validators.url(link)

def validate_link_onion(link):
    return validate_link(link) and '.onion' in link

def remove_non_alphanumeric(s):
    return re.sub(r'[^a-zA-Z0-9\s?¿!.=]+', ' ', s)

def remove_multiple_whitespace(s):
    return re.sub(r'\s+', ' ', s)

def remove_duplicates(l):
    return list(dict.fromkeys(l))

def remove_garbage(s):
    s = remove_multiple_whitespace(s)
    return remove_non_alphanumeric(s)

def remove_whitespaces(s):
    return re.sub(r'\s+', '',s)

def get_title_text(s):
    return re.search(r"Title:\s+(.*?)\n+",s).group(1)

s = "♛♛PROMOTION84%HQMDMA10g=180€♛♛|AnarchiaMarket,"
remove_non_alphanumeric(s)

def get_category_1776(s):
    return re.search(r"Category:\s+(.*?)\n+",s).group(1)

### There's a lot of data that is no well classified so for the 1776 set, the files that start with Main?category=*keyword*
<b>If the elements found on this search are zero the file will be deleted<b>
<b>If the query if Viewuser? delete the file<b>


In [ ]:
file_path = './examples/1776/'
# Function to check if the HTML content contains an element with text "0"
def contains_element_with_text_zero(file_path,file_start,id):
    for root, dirs, files in os.walk(file_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            if os.path.isfile(file_path):
                try:
                    if file_start in file_path:
                        with open(file_path, 'r', encoding='utf-8') as file:
                            html_content = file.read()
                            soup = BeautifulSoup(html_content, 'html.parser')
                            element = soup.find('span', id=id)
                            if element and element.text.strip() == '0':
                                os.remove(file_path)
                                print(f"Deleted non-HTML file: {file_path}")
                    elif  'Viewuser?' in file_path:
                        os.remove(file_path)
                        print(f"Deleted non-HTML file: {file_path}")
                    else:
                        if not file_path.endswith('.html'):
                            os.rename(file_path, file_path + ".html")
                except Exception as e:
                    print(f"Error processing {file_path}: {str(e)}")

contains_element_with_text_zero(file_path,'Main?Category','lblitemsfound')



#### Now we need to extract the categories of each item in ViewItem? and category associdated




In [ ]:
def get_categories(file_path,file_start,id):
    s = set()
    for root, dirs, files in os.walk(file_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            if os.path.isfile(file_path):
                try:
                    if file_start in file_path:
                        with open(file_path, 'r', encoding='utf-8') as file:
                            html_content = file.read()
                            soup = BeautifulSoup(html_content, 'html.parser')
                            element = soup.find('span', id=id)
                            s.add(element.text.strip())    
                            if 'General Health' in element.text:
                                print(soup.get_text())
                    else:
                        if not file_path.endswith('.html'):
                            os.rename(file_path, file_path + ".html")
                except Exception as e:
                    print(f"Error processing {file_path}: {str(e)}")
    print(s)
#get_categories('./examples/1776/','Viewitem?','lblcategory')
categories = {
    'Fraud': {'money', 'digital goods', 'electronics'},
    'Counterfeit': {'counterfeit'},
    'Drugs': {'weed','tabs','pills','pill','general health', 'oxycodone', 'hash', 'speed', 'opioids', 'drugs', 'dox', 'methamphetamine', 'shake', 'oils', 'diazepam', 'benzos', '2c-family', 'fentanyl', 'ketamine', 'other', 'white', 'ethylphenidate', 'heroin', 'synthetic', 'cannabis', 'nbome', 'dmt', 'cocaine', 'lsd', 'shrooms', 'dissociatives', 'alprazolam', 'mephedrone', 'tramadol', 'wax', 'mdma', 'prescription'},
    'Services': {'services'}
}

def match_category(category:str):
    matching_category = None
    for key, value in categories.items():
        if category.lower() in value:
            return key
    return "Drugs"
            

def categorizer_1776(file):
    soup = BeautifulSoup(file, 'html.parser')
    form_action = soup.find('form', {'method': 'post', 'action': 'Main?category=analgesics', 'id': 'form1'})
    element = soup.find('span', id='lblcategory')  
    if form_action:
        action_value = form_action['action']
        # Extract the word after 'category='
        category = action_value.split('category=')[1]
        return match_category(category)
    elif element:
        return match_category(element.text)
    else:
        return "Drugs"


In [ ]:
## 
directory = './examples/1776/'
for root, dirs, files in os.walk(directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            if os.path.isfile(file_path):
                try:
                    if 'HTML' in guess_file_type(file_path):
                        print(categorizer_1776(read_file(file_path)))
                        #print(f"Deleted non-HTML file: {file_path}")
                except Exception as e:
                    print(f"Error processing {file_path}: {str(e)}")

## Panacea categorization
We need to remove the useless htmls


This webpage is entirely based on cannabis products

In [ ]:
def filter_panacea(file):
    soup = BeautifulSoup(file, 'html.parser')
    # Find all <h3> elements in the HTML
    h3_elements = soup.find_all('h3')
    # Check if any <h3> element has the specified text
    desired_text = "No products found matching this criteria"
    foundh3 = any(h3.get_text(strip=True) == desired_text for h3 in h3_elements)
    # Find all <p> elements in the HTML
    p_elements = soup.find_all('p')
    # Check if any <p> element has the specified text
    desired_p_text = "There is no feedback found for this vendor"
    found_p = any(p.get_text(strip=True) == desired_p_text for p in p_elements)
    if found_p or foundh3:
        return False
    return True

def categorize_panacea(s):
    return 'Drugs'



In [ ]:
## 
directory = './examples/panacea/'
for root, dirs, files in os.walk(directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            if os.path.isfile(file_path):
                try:
                    if 'HTML' in guess_file_type(file_path):
                        if filter_panacea(read_file(file_path)):
                            print(file_path)
                        #print(f"Deleted non-HTML file: {file_path}")
                except Exception as e:
                    print(f"Error processing {file_path}: {str(e)}")

In [ ]:
#def valid_content():  #Remove garbage values from the content like keys, endoded data etc.
content = read_file('./examples/assassinationmarket/2014-01-15/index.html') #0cf7372fb9393cb0
e = Extractor(content)
e.extract_content_raw()
print(e.content_raw)
parsed = remove_garbage(e.content_raw)
print(parsed)
print(e.title)

In [ ]:
import os
import threading

mime = magic.Magic()

def write_to_file(file_name, data, mode='w'):
    """
    Writes data to a file, overwriting the existing content if the file already exists.

    :param file_name: The path to the file where data will be written.
    :type file_name: str

    :param data: The data to be written to the file.
    :type data: str
    """
    if mode == 'w' or mode == 'a':
        with open(file_name, mode) as file:
            file.write(data)
            file.close()
    

def process_file(file_path, categorizer, csv_list,filter):
    try:
        file = read_file(file_path)  # Define the 'read_file' function appropriately
        if filter(file_path):
            e = Extractor(file)  # Define the 'Extractor' class appropriately
            e.extract_content_raw()
            title = remove_garbage(e.title).rstrip()
            content = remove_garbage(e.content_raw).rstrip()
            category = categorizer(file)
            csv_list.append(f"{title},{content},{category}\n")
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")

def traverse_and_read(directory, categorizer, mode, f,num_threads=4):
    csv_list = ["title,content,category\n"] if mode == 'w' else [""]
    file_paths = []

    for root, dirs, files in os.walk(directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            if os.path.isfile(file_path):
                file_paths.append((file_path, categorizer, csv_list,f))

    threads = []
    for _ in range(num_threads):
        thread = threading.Thread(target=process_file_threaded, args=(file_paths,))
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    write_to_file('./examples/categorized.csv', ''.join(csv_list), mode)

def process_file_threaded(file_paths):
    while file_paths:
        file_path, category, csv_list,f = file_paths.pop()
        process_file(file_path, category, csv_list,f)


#### The process of categorization will issue a cell for each web page, since we do not have so many we can handle assigning the category manually
We may have to read some of the scraper html 
Folder ./examples/1776/2014-06-22/ removed since it contained profiles

In [ ]:
#1776
def filter_1776(s):
    return True
start_directory = './examples/1776/'
traverse_and_read(start_directory,categorizer_1776,'w',filter_1776,8)
start_directory = './examples/panacea/'

traverse_and_read(start_directory,categorize_panacea,'a',filter_panacea,8)


### Pedo funding
<b>Given the disgusting of the material I just took the input without filter<b>


In [ ]:
start_directory = './examples/pedofunding'
def categorize_pedo(s):
    return "Pedophilia"

def filter_pedo(s):
    return True

traverse_and_read(start_directory,categorize_pedo,'a',filter_pedo,8)


### Tortuga2
It is only drug market

In [ ]:
start_directory = './examples/tortuga2/'

def categorize_tortuga2(s):
    return 'Drugs'

def filter_tortuge2(s):
    return True

traverse_and_read(start_directory,categorize_tortuga2,'a',filter_tortuge2,8)

In [ ]:
import pandas as pd

# Replace 'your_dataset.csv' with the actual path to your CSV file
csv_file = './examples/categorized.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_file)

# Assuming your CSV file has a column named 'category' where the categories are stored
if 'category' in df.columns:
    # Get unique categories and convert them to a list
    unique_categories = df['category'].unique().tolist()

    # Print the unique categories
    for category in unique_categories:
        print(category)
else:
    print("The 'category' column does not exist in the CSV file.")